# Multiclass Classification using CMU dataset

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [2]:
all_results = []
def evaluate_model(model_name, y_true, y_pred):
    acc  = accuracy_score(y_true, y_pred) * 100
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0) * 100 
    f1   = f1_score(y_true, y_pred, average='weighted', zero_division=0) * 100
    all_results.append({
        'Model': model_name,
        'Accuracy (%)': round(acc, 2),  #the ratio of correctly classified samples to total samples. 
        'Precision (%)': round(prec, 2),   #for a given class, precision is the fraction of predicted positives that are truly positive.
        'F1 Score (%)': round(f1, 2)   #The harmonic mean punishes extreme imbalances between precision and recall more than the arithmetic mean would. It is the standard single metric for multi-class classification evaluation.
    })

## Data Loading

In [3]:
df = pd.read_csv("DSL-StrongPasswordData.csv")

In [4]:
df.head()

,subject,sessionIndex,rep,H.period,DD.period.t,UD.period.t,H.t,DD.t.i,UD.t.i,H.i,...,H.a,DD.a.n,UD.a.n,H.n,DD.n.l,UD.n.l,H.l,DD.l.Return,UD.l.Return,H.Return
0,s002,1,1,0.1491,0.3979,0.2488,0.1069,0.1674,0.0605,0.1169,...,0.1349,0.1484,0.0135,0.0932,0.3515,0.2583,0.1338,0.3509,0.2171,0.0742
1,s002,1,2,0.1111,0.3451,0.2340,0.0694,0.1283,0.0589,0.0908,...,0.1412,0.2558,0.1146,0.1146,0.2642,0.1496,0.0839,0.2756,0.1917,0.0747
2,s002,1,3,0.1328,0.2072,0.0744,0.0731,0.1291,0.0560,0.0821,...,0.1621,0.2332,0.0711,0.1172,0.2705,0.1533,0.1085,0.2847,0.1762,0.0945
3,s002,1,4,0.1291,0.2515,0.1224,0.1059,0.2495,0.1436,0.1040,...,0.1457,0.1629,0.0172,0.0866,0.2341,0.1475,0.0845,0.3232,0.2387,0.0813
4,s002,1,5,0.1249,0.2317,0.1068,0.0895,0.1676,0.0781,0.0903,...,0.1312,0.1582,0.0270,0.0884,0.2517,0.1633,0.0903,0.2517,0.1614,0.0818


In [5]:
df.shape

(20400, 34)

In [6]:
df.columns

Index(['subject', 'sessionIndex', 'rep', 'H.period', 'DD.period.t',
       'UD.period.t', 'H.t', 'DD.t.i', 'UD.t.i', 'H.i', 'DD.i.e', 'UD.i.e',
       'H.e', 'DD.e.five', 'UD.e.five', 'H.five', 'DD.five.Shift.r',
       'UD.five.Shift.r', 'H.Shift.r', 'DD.Shift.r.o', 'UD.Shift.r.o', 'H.o',
       'DD.o.a', 'UD.o.a', 'H.a', 'DD.a.n', 'UD.a.n', 'H.n', 'DD.n.l',
       'UD.n.l', 'H.l', 'DD.l.Return', 'UD.l.Return', 'H.Return'],
      dtype='object')

In [7]:
df.dtypes

subject             object
sessionIndex         int64
rep                  int64
H.period           float64
DD.period.t        float64
UD.period.t        float64
H.t                float64
DD.t.i             float64
UD.t.i             float64
H.i                float64
DD.i.e             float64
UD.i.e             float64
H.e                float64
DD.e.five          float64
UD.e.five          float64
H.five             float64
DD.five.Shift.r    float64
UD.five.Shift.r    float64
H.Shift.r          float64
DD.Shift.r.o       float64
UD.Shift.r.o       float64
H.o                float64
DD.o.a             float64
UD.o.a             float64
H.a                float64
DD.a.n             float64
UD.a.n             float64
H.n                float64
DD.n.l             float64
UD.n.l             float64
H.l                float64
DD.l.Return        float64
UD.l.Return        float64
H.Return           float64
dtype: object

In [8]:
df.isnull().sum()

subject            0
sessionIndex       0
rep                0
H.period           0
DD.period.t        0
UD.period.t        0
H.t                0
DD.t.i             0
UD.t.i             0
H.i                0
DD.i.e             0
UD.i.e             0
H.e                0
DD.e.five          0
UD.e.five          0
H.five             0
DD.five.Shift.r    0
UD.five.Shift.r    0
H.Shift.r          0
DD.Shift.r.o       0
UD.Shift.r.o       0
H.o                0
DD.o.a             0
UD.o.a             0
H.a                0
DD.a.n             0
UD.a.n             0
H.n                0
DD.n.l             0
UD.n.l             0
H.l                0
DD.l.Return        0
UD.l.Return        0
H.Return           0
dtype: int64

In [9]:
df['subject'].value_counts()

subject
s002    400
s044    400
s034    400
s035    400
s036    400
s037    400
s038    400
s039    400
s040    400
s041    400
s042    400
s043    400
s046    400
s032    400
s047    400
s048    400
s049    400
s050    400
s051    400
s052    400
s053    400
s054    400
s055    400
s056    400
s033    400
s031    400
s003    400
s017    400
s004    400
s005    400
s007    400
s008    400
s010    400
s011    400
s012    400
s013    400
s015    400
s016    400
s018    400
s030    400
s019    400
s020    400
s021    400
s022    400
s024    400
s025    400
s026    400
s027    400
s028    400
s029    400
s057    400
Name: count, dtype: int64

In [10]:
# LabelEncoder is a preprocessing utility which Converts string class labels into integers. Internally builds a sorted mapping, alphabetical order such as s002→0, s003→1, s044→50 etc.
# Must be done because all sklearn models require numeric targets. 
X = df.drop(['subject'], axis=1)
le = LabelEncoder()
y_encoded = le.fit_transform(df['subject'])

In [11]:
# Randomly shuffles the dataset and splits it into training and test subsets.
X_train, X_test, y_train_encoded, y_test_encoded = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Logistic Regression

In [12]:
#  It models the probability that a sample belongs to a class using the logistic (sigmoid) function.
model = LogisticRegression(max_iter=5000) # gives the solver enough steps to converge across all 51 binary problems.
model.fit(X_train, y_train_encoded)

LogisticRegression(max_iter=5000)

In [13]:
y_pred = model.predict(X_test)
evaluate_model("Logistic Regression (unscaled)", y_test_encoded, y_pred)

In [14]:
# Evaluation Metrics
cm = confusion_matrix(y_test_encoded, y_pred)
print(cm)
print("accuracy score: ", accuracy_score(y_test_encoded, y_pred) * 100, "%")
print("precision score: ", precision_score(y_test_encoded, y_pred, average='weighted') * 100, "%")
print("recall score: ", recall_score(y_test_encoded, y_pred, average='weighted') * 100, "%")
print("F1 score: ", f1_score(y_test_encoded, y_pred, average='weighted') * 100, "%")

[[45  0  7 ...  0  1  0]
 [ 0 66  0 ...  0  0  0]
 [ 1  1 55 ...  0  0  0]
 ...
 [ 1  0  0 ... 70  3  0]
 [ 0  1  0 ...  0 62  3]
 [ 0  0  0 ...  0  0 54]]
accuracy score:  74.0686274509804 %
precision score:  74.116542070719 %
recall score:  74.0686274509804 %
F1 score:  73.32594945996482 %


In [15]:
# Scaling -> Standardizes each feature to mean=0 and standard deviation=1 using the formula z = (x − μ) / σ
# It must be fit only on training data (fit_transform) and then applied to test data (transform) to avoid data leakage.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [16]:
model = LogisticRegression(max_iter=5000)
model.fit(X_train_scaled, y_train_encoded)
y_pred = model.predict(X_test_scaled)
evaluate_model("Logistic Regression (scaled)", y_test_encoded, y_pred)
print("accuracy score: ", accuracy_score(y_test_encoded, y_pred) * 100, "%")
print("precision score: ", precision_score(y_test_encoded, y_pred, average='weighted') * 100, "%")
print("recall score: ", recall_score(y_test_encoded, y_pred, average='weighted') * 100, "%")
print("F1 score: ", f1_score(y_test_encoded, y_pred, average='weighted') * 100, "%")

accuracy score:  85.0 %
precision score:  84.93421120919976 %
recall score:  85.0 %
F1 score:  84.86066425586188 %


# Naive Bayes

In [17]:
# "Naive" = assumes all features are conditionally independent given the class, which simplifies the joint probability into a product. 
# "Gaussian" = assumes each feature follows a normal distribution per class, estimating μ and σ from training data.
# Extremely fast, no hyperparameters, but the independence assumption is almost never perfectly true in real data.

gnb_model = GaussianNB()
gnb_model.fit(X_train, y_train_encoded)

GaussianNB()

In [18]:
y_pred = gnb_model.predict(X_test)
evaluate_model("Naive Bayes", y_test_encoded, y_pred)

In [19]:
print("accuracy score: ", accuracy_score(y_test_encoded, y_pred) * 100, "%")
print("precision score: ", precision_score(y_test_encoded, y_pred, average='weighted') * 100, "%")
print("recall score: ", recall_score(y_test_encoded, y_pred, average='weighted') * 100, "%")
print("F1 score: ", f1_score(y_test_encoded, y_pred, average='weighted') * 100, "%")

accuracy score:  67.47549019607844 %
precision score:  69.36789589223913 %
recall score:  67.47549019607844 %
F1 score:  67.05668586325312 %


# KNN

In [20]:
# it stores the entire training set and builds no model during training. 
# At prediction time, computes Euclidean distance from the query point to every training point, selects the k nearest, and assigns the majority class label

classifier = KNeighborsClassifier()
param_grid = {"n_neighbors": [3, 5, 7, 9]}

# k-fold cross-validation : training data is divided into k equal folds. The model is trained k times, each time holding out one different fold as validation. The k scores are averaged. Gives more reliable hyperparameter selection than a single train/val split.
# GridSearchCV: Exhaustively trains and evaluates a model for every combination in a parameter grid using k-fold cross-validation.
classifierCV = GridSearchCV(
    classifier,
    param_grid,
    cv=5, # means 5-fold CV.
    scoring="accuracy" #  metric used to rank combinations.
)

classifierCV.fit(X_train_scaled, y_train_encoded)

y_pred = classifierCV.predict(X_test_scaled)
evaluate_model(f"KNN (k={classifierCV.best_params_['n_neighbors']})", y_test_encoded, y_pred)

In [21]:
print("accuracy score: ", accuracy_score(y_test_encoded, y_pred) * 100, "%")
print("precision score: ", precision_score(y_test_encoded, y_pred, average='weighted') * 100, "%")
print("recall score: ", recall_score(y_test_encoded, y_pred, average='weighted') * 100, "%")
print("F1 score: ", f1_score(y_test_encoded, y_pred, average='weighted') * 100, "%")

# results
res = pd.DataFrame(classifierCV.cv_results_)
print(res[["param_n_neighbors", "mean_test_score"]])
print("Best params:", classifierCV.best_params_)

accuracy score:  86.00490196078432 %
precision score:  86.8631863639289 %
recall score:  86.00490196078432 %
F1 score:  85.98587141676595 %
   param_n_neighbors  mean_test_score
0                  3         0.843627
1                  5         0.850919
2                  7         0.849081
3                  9         0.846691
Best params: {'n_neighbors': 5}


# Decision Tree

In [22]:
# Recursively partitions the feature space by selecting, at each node, the feature and threshold that minimizes Gini impurity. Builds top-down, greedily. 

final_dt = DecisionTreeClassifier(max_depth=15, random_state=42)
final_dt.fit(X_train, y_train_encoded)
y_pred = final_dt.predict(X_test)
evaluate_model("Decision Tree (depth=15)", y_test_encoded, y_pred)

print("--- Final Decision Tree Results ---")
print("accuracy score: ",  accuracy_score(y_test_encoded, y_pred) * 100, "%")
print("precision score: ", precision_score(y_test_encoded, y_pred, average='weighted', zero_division=0) * 100, "%")
print("F1 score: ",        f1_score(y_test_encoded, y_pred, average='weighted', zero_division=0) * 100, "%")

--- Final Decision Tree Results ---
accuracy score:  68.77450980392157 %
precision score:  71.67319994730804 %
F1 score:  69.54532714069977 %


# Decision Tree With Pre-Pruning

In [23]:
#  restricts growth during training. max_depth caps tree height.
max_depths = [5, 6, 7, 8, 9, 10, 15, 20, 25, 30]  
min_samples_splits = [5, 10, 15, 20, 25, 30] # ets the minimum number of samples a node must have before it can be split. 
results = []

for depth in max_depths:
    for split in min_samples_splits:
        model = DecisionTreeClassifier(
            max_depth=depth,           
            min_samples_split=split,  
            random_state=42
        )
        model.fit(X_train, y_train_encoded)
        y_pred = model.predict(X_test)

        acc  = accuracy_score(y_test_encoded, y_pred) * 100
        prec = precision_score(y_test_encoded, y_pred, average='weighted', zero_division=0) * 100  
        rec  = recall_score(y_test_encoded, y_pred, average='weighted', zero_division=0) * 100
        f1   = f1_score(y_test_encoded, y_pred, average='weighted', zero_division=0) * 100

        results.append({
            "max_depth": depth,
            "min_samples_split": split,
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1": f1
        })
        print(f"depth={depth} | split={split} | accuracy={acc:.2f}% | precision={prec:.2f}% | recall={rec:.2f}% | F1={f1:.2f}%")


results_df = pd.DataFrame(results)
best = results_df.loc[results_df['accuracy'].idxmax()]
print(f"\nBest depth: {int(best['max_depth'])} | Best split: {int(best['min_samples_split'])} | Accuracy: {best['accuracy']:.2f}%")

best_dt = DecisionTreeClassifier(
    max_depth=int(best['max_depth']),
    min_samples_split=int(best['min_samples_split']),
    random_state=42
)
best_dt.fit(X_train, y_train_encoded)
y_pred = best_dt.predict(X_test)
evaluate_model(f"DT Pre-Pruning (depth={int(best['max_depth'])}, split={int(best['min_samples_split'])})", y_test_encoded, y_pred)  

depth=5 | split=5 | accuracy=28.82% | precision=24.89% | recall=28.82% | F1=23.62%
depth=5 | split=10 | accuracy=28.82% | precision=24.89% | recall=28.82% | F1=23.62%
depth=5 | split=15 | accuracy=28.82% | precision=25.36% | recall=28.82% | F1=23.59%
depth=5 | split=20 | accuracy=28.82% | precision=25.36% | recall=28.82% | F1=23.59%
depth=5 | split=25 | accuracy=28.82% | precision=25.36% | recall=28.82% | F1=23.59%
depth=5 | split=30 | accuracy=28.82% | precision=25.36% | recall=28.82% | F1=23.59%
depth=6 | split=5 | accuracy=36.45% | precision=39.33% | recall=36.45% | F1=33.26%
depth=6 | split=10 | accuracy=36.40% | precision=39.26% | recall=36.40% | F1=33.20%
depth=6 | split=15 | accuracy=36.37% | precision=39.16% | recall=36.37% | F1=33.17%
depth=6 | split=20 | accuracy=36.32% | precision=39.09% | recall=36.32% | F1=33.11%
depth=6 | split=25 | accuracy=36.32% | precision=39.09% | recall=36.32% | F1=33.11%
depth=6 | split=30 | accuracy=36.30% | precision=39.06% | recall=36.30% | F1=3

# Decision Tree with Post-Pruning

In [24]:
# grows a full unconstrained tree first, then prunes it backward using ccp_alpha. 
model = DecisionTreeClassifier(random_state=42)
path = model.cost_complexity_pruning_path(X_train, y_train_encoded)
ccp_alphas = path.ccp_alphas[:-1]

print(f"Total alpha values to try: {len(ccp_alphas)}")

Total alpha values to try: 2135


In [25]:
ccp_alphas_sampled = np.linspace(ccp_alphas.min(), ccp_alphas.max(), 20)

print(f"Sampled alpha values to try: {len(ccp_alphas_sampled)}")
print(ccp_alphas_sampled)

Sampled alpha values to try: 20
[0.         0.00067808 0.00135617 0.00203425 0.00271234 0.00339042
 0.00406851 0.00474659 0.00542468 0.00610276 0.00678085 0.00745893
 0.00813702 0.0088151  0.00949318 0.01017127 0.01084935 0.01152744
 0.01220552 0.01288361]


In [26]:
results = []
for alpha in ccp_alphas_sampled:
    model = DecisionTreeClassifier(random_state=42, ccp_alpha=alpha)
    model.fit(X_train, y_train_encoded)
    y_pred = model.predict(X_test)

    acc  = accuracy_score(y_test_encoded, y_pred) * 100
    prec = precision_score(y_test_encoded, y_pred, average='weighted', zero_division=0) * 100
    rec  = recall_score(y_test_encoded, y_pred, average='weighted', zero_division=0) * 100
    f1   = f1_score(y_test_encoded, y_pred, average='weighted', zero_division=0) * 100

    results.append({
        "ccp_alpha": round(alpha, 6),
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1
    })
    print(f"alpha={alpha:.6f} | accuracy={acc:.2f}% | precision={prec:.2f}% | recall={rec:.2f}% | F1={f1:.2f}%")

results_df = pd.DataFrame(results)
best = results_df.loc[results_df['accuracy'].idxmax()]
print(f"\nBest alpha: {best['ccp_alpha']} | Accuracy: {best['accuracy']:.2f}%")

alpha=0.000000 | accuracy=72.89% | precision=73.23% | recall=72.89% | F1=72.93%
alpha=0.000678 | accuracy=65.81% | precision=68.36% | recall=65.81% | F1=66.16%
alpha=0.001356 | accuracy=60.76% | precision=64.97% | recall=60.76% | F1=61.34%
alpha=0.002034 | accuracy=57.79% | precision=63.58% | recall=57.79% | F1=58.21%
alpha=0.002712 | accuracy=55.59% | precision=56.83% | recall=55.59% | F1=54.43%
alpha=0.003390 | accuracy=52.45% | precision=52.96% | recall=52.45% | F1=50.66%
alpha=0.004069 | accuracy=49.46% | precision=48.65% | recall=49.46% | F1=46.75%
alpha=0.004747 | accuracy=46.18% | precision=44.49% | recall=46.18% | F1=42.98%
alpha=0.005425 | accuracy=45.20% | precision=42.25% | recall=45.20% | F1=41.12%
alpha=0.006103 | accuracy=42.30% | precision=36.87% | recall=42.30% | F1=37.21%
alpha=0.006781 | accuracy=38.09% | precision=31.49% | recall=38.09% | F1=32.47%
alpha=0.007459 | accuracy=36.00% | precision=28.06% | recall=36.00% | F1=29.69%
alpha=0.008137 | accuracy=33.33% | preci

In [27]:
best_alpha = best['ccp_alpha']
final_model = DecisionTreeClassifier(random_state=42, ccp_alpha=best_alpha)
final_model.fit(X_train, y_train_encoded)
y_pred = final_model.predict(X_test)

evaluate_model("Decision Tree (Post-Pruning)", y_test_encoded, y_pred) 

print("accuracy score: ",  accuracy_score(y_test_encoded, y_pred) * 100, "%")
print("precision score: ", precision_score(y_test_encoded, y_pred, average='weighted', zero_division=0) * 100, "%")
print("recall score: ",    recall_score(y_test_encoded, y_pred, average='weighted', zero_division=0) * 100, "%")
print("F1 score: ",        f1_score(y_test_encoded, y_pred, average='weighted', zero_division=0) * 100, "%")

accuracy score:  72.8921568627451 %
precision score:  73.23250221494385 %
recall score:  72.8921568627451 %
F1 score:  72.929948965037 %


# SVM

In [28]:
# SVM finds the maximum-margin hyperplane separating classes.

C_values = [0.1, 1, 10, 100]
results = []

for c in C_values:   # C is the regularization parameter
    svm_model = SVC(kernel='rbf', C=c, random_state=42)
    svm_model.fit(X_train_scaled, y_train_encoded) 
    y_pred = svm_model.predict(X_test_scaled)

    acc  = accuracy_score(y_test_encoded, y_pred) * 100
    prec = precision_score(y_test_encoded, y_pred, average='weighted', zero_division=0) * 100
    rec  = recall_score(y_test_encoded, y_pred, average='weighted', zero_division=0) * 100
    f1   = f1_score(y_test_encoded, y_pred, average='weighted', zero_division=0) * 100

    results.append({"C": c, "accuracy": acc, "precision": prec, "recall": rec, "f1": f1})
    print(f"C={c} | accuracy={acc:.2f}% | precision={prec:.2f}% | recall={rec:.2f}% | F1={f1:.2f}%")

results_df = pd.DataFrame(results)
best = results_df.loc[results_df['accuracy'].idxmax()]
print(f"\nBest C: {best['C']} | Accuracy: {best['accuracy']:.2f}%")

# Step 2 — Train final model with best C
best_c = best['C']
final_svm = SVC(kernel='rbf', C=best_c, random_state=42)
final_svm.fit(X_train_scaled, y_train_encoded)
y_pred = final_svm.predict(X_test_scaled)
evaluate_model(f"SVM (C={best_c}, rbf)", y_test_encoded, y_pred)

print("\n--- Final SVM Results ---")
print("accuracy score: ",  accuracy_score(y_test_encoded, y_pred) * 100, "%")
print("precision score: ", precision_score(y_test_encoded, y_pred, average='weighted', zero_division=0) * 100, "%")
print("recall score: ",    recall_score(y_test_encoded, y_pred, average='weighted', zero_division=0) * 100, "%")
print("F1 score: ",        f1_score(y_test_encoded, y_pred, average='weighted', zero_division=0) * 100, "%")

C=0.1 | accuracy=82.52% | precision=85.30% | recall=82.52% | F1=83.25%
C=1 | accuracy=90.17% | precision=90.57% | recall=90.17% | F1=90.22%
C=10 | accuracy=91.15% | precision=91.38% | recall=91.15% | F1=91.17%
C=100 | accuracy=91.23% | precision=91.42% | recall=91.23% | F1=91.23%

Best C: 100.0 | Accuracy: 91.23%

--- Final SVM Results ---
accuracy score:  91.22549019607843 %
precision score:  91.41872981236986 %
recall score:  91.22549019607843 %
F1 score:  91.22986812798294 %


# Random Forest

In [29]:
# RF an ensemble method that builds multiple decision trees using two sources of randomness: (1) bootstrap sampling — each tree is trained on a random sample with replacement from the training set; (2) feature randomness —> at each node split, only a random subset of features is considered. Final prediction is by majority vote. 
rf = RandomForestClassifier(
    n_estimators=1001,
    oob_score=True,
    max_depth=15,       
    random_state=42
)

rf.fit(X_train, y_train_encoded)

y_pred_train = rf.predict(X_train)
y_pred_test  = rf.predict(X_test)
evaluate_model("Random Forest", y_test_encoded, y_pred_test)

# OOB Score (out-of-bag score) is a built-in validation: each tree is evaluated on the ~37% of samples it never saw during its bootstrap training, giving a free generalization estimate without a separate validation set.
print("OOB Score: ", rf.oob_score_ * 100, "%")

# Train vs Test (check overfitting)
print("Training accuracy: ", accuracy_score(y_train_encoded, y_pred_train) * 100, "%")
print("Testing accuracy:  ", accuracy_score(y_test_encoded, y_pred_test) * 100, "%")

# Full evaluation
print("\n--- Random Forest Results ---")
print("accuracy score: ", accuracy_score(y_test_encoded, y_pred_test) * 100, "%")
print("precision score: ",precision_score(y_test_encoded, y_pred_test, average='weighted', zero_division=0) * 100, "%")
print("F1 score: ",f1_score(y_test_encoded, y_pred_test, average='weighted', zero_division=0) * 100, "%")

OOB Score:  92.89828431372548 %
Training accuracy:  99.86519607843137 %
Testing accuracy:   93.70098039215686 %

--- Random Forest Results ---
accuracy score:  93.70098039215686 %
precision score:  93.87916157850853 %
F1 score:  93.67641363591342 %


# XgBoost

In [30]:
# Extreme Gradient Boosting -> An ensemble method where trees are built sequentially, each one trained to correct the residual errors of all previous trees combined.

xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    eval_metric='mlogloss'  #required for multiclass
)
xgb_model.fit(X_train, y_train_encoded)
y_pred = xgb_model.predict(X_test)
evaluate_model("XGBoost", y_test_encoded, y_pred)

print("accuracy score: ",  accuracy_score(y_test_encoded, y_pred) * 100, "%")
print("precision score: ", precision_score(y_test_encoded, y_pred, average='weighted', zero_division=0) * 100, "%")
print("F1 score: ",        f1_score(y_test_encoded, y_pred, average='weighted', zero_division=0) * 100, "%")

accuracy score:  93.99509803921569 %
precision score:  94.13421187312449 %
F1 score:  94.00160226608634 %


In [31]:
# Final Results Table
results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values('Accuracy (%)', ascending=False).reset_index(drop=True)
results_df.index += 1
results_df.index.name = 'Rank'

print("=" * 70)
print("MODEL COMPARISON - KEYSTROKE DYNAMICS CLASSIFICATION")
print("=" * 70)
print(results_df.to_string())
print("=" * 70)
print(f"Best Model : {results_df.iloc[0]['Model']}")
print(f"Accuracy   : {results_df.iloc[0]['Accuracy (%)']:.2f}%")
print(f"Precision  : {results_df.iloc[0]['Precision (%)']:.2f}%")
print(f"F1 Score   : {results_df.iloc[0]['F1 Score (%)']:.2f}%")
print("=" * 70)

      MODEL COMPARISON - KEYSTROKE DYNAMICS CLASSIFICATION
                                   Model  Accuracy (%)  Precision (%)  F1 Score (%)
Rank                                                                               
1                                XGBoost         94.00          94.13         94.00
2                          Random Forest         93.70          93.88         93.68
3                     SVM (C=100.0, rbf)         91.23          91.42         91.23
4                              KNN (k=5)         86.00          86.86         85.99
5           Logistic Regression (scaled)         85.00          84.93         84.86
6         Logistic Regression (unscaled)         74.07          74.12         73.33
7           Decision Tree (Post-Pruning)         72.89          73.23         72.93
8     DT Pre-Pruning (depth=25, split=5)         71.86          72.62         72.01
9               Decision Tree (depth=15)         68.77          71.67         69.55
10               